# India DPDP Act - GraphRAG Assistant with Human-in-the-Loop Review

A working demo of a policy knowledge-graph + retrieval-augmented assistant, scoped to a single law
(India's Digital Personal Data Protection Act, 2023) so the full pipeline is easy to run, explain,
and verify end to end. Runs entirely in Google Colab, fully open source, no API key needed anywhere.

## What this demonstrates

A multi-country policy assistant needs: a **knowledge graph** connecting laws to their obligations,
rights, and penalties; a **vector store** for semantic search over the legal text; **AI agents** that
retrieve and generate answers; a **human-in-the-loop review gate** so nothing unverified reaches an
end user; and an interface (here, a chat app) for people to actually use it. This notebook builds all
of that for one law, as a working proof of concept for the larger project.

## Architecture

```
Official DPDP Act PDF (fetched live from meity.gov.in)
        |
        v
Parse into 44 numbered sections (regex, validated against the real text)
        |
        v
Rule-based tagging: category (Obligation / Right / Penalty / Definition) + confidence
        |
        v
  +-----+-----------------------------------+
  |                                         |
  v                                         v
Auto-approved                      Held for HUMAN REVIEW
(safe category, high confidence)   (touches Obligation/Penalty, or low confidence)
  |                                         |
  v                                         v
 Written to Qdrant (vectors)         Sits in a review queue until a human
 + Neo4j (graph)                     approves/rejects it via the Streamlit
  |                                  sidebar - only then does it get written
  |                                  to Qdrant + Neo4j too
  v
FastAPI backend (/ask, /pending-review, /approve-review-item)
  |
  v
On a question: vector search (Qdrant) + graph context (Neo4j) retrieved,
then Ollama (qwen2.5:3b, local, no API key) generates a cited answer -
UNLESS the question itself is high-risk (penalty/obligation/cross-border),
in which case it's also held for human review instead of answered directly
  |
  v
Streamlit chat UI  --->  Cloudflare quick tunnel  --->  public HTTPS URL
```

## Tech stack (all open source, all free, no signup required anywhere)

| Component | Tool | Role |
|---|---|---|
| Knowledge graph | Neo4j (Community) | Sections linked to obligations, rights, penalties, definitions |
| Vector store | Qdrant (in-memory) | Semantic search over section text |
| Embeddings | `sentence-transformers` (`all-MiniLM-L6-v2`) | Turns section text and questions into vectors |
| Local LLM | Ollama, running `qwen2.5:3b` | Generates the final answer text - only called per question, not during ingestion |
| Backend | FastAPI + Uvicorn | `/ask`, `/pending-review`, `/approve-review-item`, `/health` |
| Frontend | Streamlit | Chat interface + live human-review sidebar |
| Public URL | `cloudflared` quick tunnel | Free, no signup, no interstitial warning page |
| Source data | Official Gazette PDF, Ministry of Electronics & IT (MeitY) | Fetched live at runtime, not hardcoded |

## Human-in-the-loop: two checkpoints, and why they matter (with a real example)

**Checkpoint 1 - before anything enters the graph/vector store.** Every parsed section is tagged by
chapter and title (e.g. Chapter II = "Obligations of Data Fiduciary"). Any section touching an
Obligation or Penalty, or with a low parse-confidence score, is held out of Qdrant/Neo4j entirely
until a human approves it through the Streamlit sidebar. This is `if Claude proposes, a human
disposes` applied to policy data: a wrong reading of a penalty clause is worse than a missing one.

**Checkpoint 2 - before a generated answer is shown.** Questions containing high-risk keywords
(penalty, obligation, breach, cross-border transfer) never get an auto-generated answer - they're
returned as `pending_review` with the retrieved context attached, so a human decides before anything
resembling advice reaches an end user.

**Why this isn't just theoretical - an example from testing this exact notebook:** asking the
one-word query *"rules"* correctly retrieved Section 41 (citations are computed separately from the
generated text, via vector search), but the local model's generated sentence mislabeled it as
"Section 42" while paraphrasing. Asking the more specific *"what does section 41 require"* produced
an accurate answer citing the same section correctly. This is a real, observed limitation of small
local LLMs: they can retrieve the right source and still misstate a detail while summarizing it -
which is exactly why citations are surfaced independently of the generated prose, and why
high-risk answers are gated behind human review rather than trusted automatically.

## How to run

Run All, top to bottom. Cells 2-3 install and start Neo4j and Ollama (a few minutes, one-time per
session). Cell 8 waits for the backend and runs a live test - it should complete in well under a
minute, since no LLM calls happen until a question is actually asked. Cell 9 prints a public HTTPS
URL (`*.trycloudflare.com`) - open it directly, no password or signup needed.

## Performance notes

Generation runs on Ollama's CPU inference by default on Colab's free tier, which is the main source
of per-question latency. Three things help:
- **Switch to a GPU runtime** (Runtime -> Change runtime type -> T4 GPU) before running - Ollama
  auto-detects CUDA, typically a 5-10x speedup, zero code changes needed.
- The model is **pre-warmed** right after it's pulled (Section 3), so the first real question isn't
  also paying the one-time cost of loading weights into memory.
- Generation is capped (`num_predict`) and the model is kept loaded between calls (`keep_alive`), and
  identical repeat questions are served from an in-memory cache instead of re-running generation.

## Known limitations (worth stating plainly, not hiding)

- **Tagging is rule-based (chapter/title keyword matching), not LLM-based**, for reliability and
  speed in a live demo - see "Where this still needs to grow" at the end for how a real extraction
  agent would replace this.
- **Only the original 2023 Act text is covered** - the Digital Personal Data Protection Rules, 2025
  (notified 13 November 2025) are not included.
- **Neo4j and Qdrant are ephemeral** - both live inside this Colab session and are wiped when it ends.
- **Small local LLM can misstate details while summarizing**, even when citing the correct source -
  demonstrated above. Citations should always be checked against the underlying section text.

## Source

Official Gazette of India, Ministry of Law and Justice, 11 August 2023, published by the Ministry of
Electronics and Information Technology (MeitY):
https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf


## 1. Install everything (system + Python + tunnel tooling)

In [ ]:
print("Installing system dependencies (zstd, OpenJDK for Neo4j, Ollama)...")
!apt-get update -qq
!apt-get install -y -qq zstd openjdk-17-jre-headless

!curl -fsSL https://ollama.com/install.sh | sh

print()
print("Installing cloudflared (for the public tunnel - free, no signup, no interstitial page)...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print()
print("Installing Python libraries...")
!pip install -q pypdf sentence-transformers qdrant-client neo4j ollama requests pydantic fastapi uvicorn streamlit


## 2. Start Neo4j

Downloads directly from `dist.neo4j.org` (not the `neo4j.com/artifact.php` redirect) and waits until
the database actually accepts connections before moving on.

In [ ]:
import subprocess
import time
import os

NEO4J_VERSION = "5.15.0"
NEO4J_DIR = f"/content/neo4j-community-{NEO4J_VERSION}"

print("Downloading Neo4j...")
subprocess.run(
    ["wget", "-q", f"https://dist.neo4j.org/neo4j-community-{NEO4J_VERSION}-unix.tar.gz", "-O", "neo4j.tar.gz"],
    check=True,
)
subprocess.run(["tar", "-xzf", "neo4j.tar.gz"], check=True)
os.environ["NEO4J_HOME"] = NEO4J_DIR

print("Setting initial password...")
subprocess.run([f"{NEO4J_DIR}/bin/neo4j-admin", "dbms", "set-initial-password", "password"], check=True)

print("Starting Neo4j (background)...")
subprocess.Popen([f"{NEO4J_DIR}/bin/neo4j", "start"])


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "password")


def wait_for_neo4j(max_attempts=30, delay_seconds=5):
    for attempt in range(max_attempts):
        try:
            driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
            driver.verify_connectivity()
            print(f"Neo4j is up (took about {attempt * delay_seconds}s).")
            driver.close()
            return
        except Exception:
            time.sleep(delay_seconds)
    raise RuntimeError("Neo4j did not become ready in time - re-run this cell, it may just need longer.")


wait_for_neo4j()


## 3. Start Ollama and pull the local model

Only used for answering questions later (one call per question) - not for startup ingestion, so a
slow pull here doesn't block the backend from becoming healthy.

**On speed:** if you're on Colab's free CPU-only runtime, `qwen2.5:3b` generation can take a while
per question. The single biggest speedup available for free is switching to a GPU runtime -
**Runtime -> Change runtime type -> T4 GPU** - before running this notebook. Ollama detects CUDA
automatically and uses it with no code changes needed. The cell below also pre-warms the model
(loads it into memory right now) so your *first* real question during the demo isn't slow too.

In [ ]:
print("Starting Ollama server (background)...")
subprocess.Popen(["ollama", "serve"])
time.sleep(10)

print("Pulling qwen2.5:3b (open source, ~2GB, one-time download)...")
subprocess.run(["ollama", "pull", "qwen2.5:3b"], check=True)

print("Pre-warming the model (loading it into memory now, so the first real question is fast)...")
import ollama as _ollama_warmup
_ollama_warmup.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Say ready."}],
    options={"num_predict": 5},
    keep_alive="30m",
)
print("Ollama ready and warmed up.")


## 4. Quick sanity check - live fetch + parse (no LLM, no wait)

Confirms the PDF fetch and section splitter work. Same sequential-section-number splitter used in
`main.py` below - validated to correctly find all 44 sections in the real Act text.

In [ ]:
import re
import requests
from pypdf import PdfReader

DPDP_PDF_URL = "https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf"

response = requests.get(DPDP_PDF_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
response.raise_for_status()
with open("dpdp_act_2023.pdf", "wb") as f:
    f.write(response.content)

reader = PdfReader("dpdp_act_2023.pdf")
raw_text = ""
for page in reader.pages:
    raw_text += (page.extract_text() or "") + "\n"


def find_section_start(text, number, search_from):
    pattern = re.compile(rf"(?:^|\n){number}\.\s")
    match = pattern.search(text, search_from)
    return match.start() if match else -1


positions = {}
cursor = 0
for n in range(1, 45):
    pos = find_section_start(raw_text, n, cursor)
    if pos != -1:
        positions[n] = pos
        cursor = pos + 1

missing = [n for n in range(1, 45) if n not in positions]
print(f"Fetched {len(response.content)} bytes. Parsed {len(positions)} of 44 sections.")
print("Missing:", missing if missing else "none - all 44 found")


# 5. Write the FastAPI backend (`main.py`)

Self-contained: fetches the PDF, parses all 44 sections, tags each with a **rule-based** category
check (obligations/rights/penalties/definitions, by chapter and title keyword) and a confidence
heuristic - this whole step takes seconds. Sections touching an obligation or penalty, or with low
confidence, are held back from Qdrant/Neo4j pending human approval. Ollama is only invoked inside
`/ask`, once per question. Written via `%%writefile` so there's no string-escaping risk.

In [ ]:
%%writefile main.py
# DPDP Act GraphRAG FastAPI backend
#
# Startup: fetches the official PDF, parses it into sections, tags each
# with rule-based categories (fast - no LLM calls here), and applies a
# human-in-the-loop review gate before anything is written to Qdrant/Neo4j.
# Ollama (qwen2.5:3b) is only called inside /ask, once per question.
# No API key needed anywhere. Requires Neo4j and Ollama already running.

import json
import os
import re
import uuid
from datetime import datetime, timezone
from typing import List, Optional

import requests
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from neo4j import GraphDatabase
import ollama

app = FastAPI(title="DPDP Act GraphRAG API")

DPDP_PDF_URL = "https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf"
LOCAL_PDF_PATH = "dpdp_act_2023.pdf"
COLLECTION = "dpdp_act_chunks"
MAX_SECTIONS = 44  # rule-based tagging is fast, so the whole Act is covered by default
CONFIDENCE_THRESHOLD = 0.85
SENSITIVE_CATEGORIES = ("obligations", "penalties")
HIGH_RISK_QUERY_KEYWORDS = (
    "penalty", "fine", "punish", "breach", "obligation", "must",
    "cross-border", "cross border", "transfer outside",
)

embedder = SentenceTransformer("all-MiniLM-L6-v2")
qdrant = QdrantClient(":memory:")
neo4j_driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))

REVIEW_QUEUE = {}
ANSWER_CACHE = {}  # simple exact-match cache: repeat questions skip retrieval + generation entirely
ALL_SECTIONS = {}

CHAPTER_MAP = [
    ("I", "PRELIMINARY", 1, 3),
    ("II", "OBLIGATIONS OF DATA FIDUCIARY", 4, 10),
    ("III", "RIGHTS AND DUTIES OF DATA PRINCIPAL", 11, 15),
    ("IV", "SPECIAL PROVISIONS", 16, 17),
    ("V", "DATA PROTECTION BOARD OF INDIA", 18, 26),
    ("VI", "POWERS, FUNCTIONS AND PROCEDURE TO BE FOLLOWED BY BOARD", 27, 28),
    ("VII", "APPEAL AND ALTERNATE DISPUTE RESOLUTION", 29, 32),
    ("VIII", "PENALTIES AND ADJUDICATION", 33, 34),
    ("IX", "MISCELLANEOUS", 35, 44),
]

SECTION_TITLES = {
    1: "Short title and commencement", 2: "Definitions", 3: "Application of Act",
    4: "Grounds for processing personal data", 5: "Notice", 6: "Consent",
    7: "Certain legitimate uses", 8: "General obligations of Data Fiduciary",
    9: "Processing of personal data of children",
    10: "Additional obligations of Significant Data Fiduciary",
    11: "Right to access information about personal data",
    12: "Right to correction and erasure of personal data",
    13: "Right of grievance redressal", 14: "Right to nominate",
    15: "Duties of Data Principal", 16: "Processing of personal data outside India",
    17: "Exemptions", 18: "Establishment of Board",
    19: "Composition and qualifications for appointment of Chairperson and Members",
    20: "Salary, allowances payable to and term of office",
    21: "Disqualifications for appointment and continuation as Chairperson and Members",
    22: "Resignation by Members and filling of vacancy", 23: "Proceedings of Board",
    24: "Officers and employees of Board", 25: "Members and officers to be public servants",
    26: "Powers of Chairperson", 27: "Powers and functions of Board",
    28: "Procedure to be followed by Board", 29: "Appeal to Appellate Tribunal",
    30: "Orders passed by Appellate Tribunal to be executable as decree",
    31: "Alternate dispute resolution", 32: "Voluntary undertaking", 33: "Penalties",
    34: "Crediting sums realised by way of penalties to Consolidated Fund of India",
    35: "Protection of action taken in good faith", 36: "Power to call for information",
    37: "Power of Central Government to issue directions", 38: "Consistency with other laws",
    39: "Bar of jurisdiction", 40: "Power to make rules",
    41: "Laying of rules and certain notifications", 42: "Power to amend Schedule",
    43: "Power to remove difficulties", 44: "Amendments to certain Acts",
}


def chapter_for_section(section_number):
    for roman, title, start, end in CHAPTER_MAP:
        if start <= section_number <= end:
            return f"Chapter {roman} - {title}"
    return "Unknown"


def find_section_start(text, number, search_from):
    pattern = re.compile(rf"(?:^|\n){number}\.\s")
    match = pattern.search(text, search_from)
    return match.start() if match else -1


def split_into_sections(text, max_section=44):
    positions = {}
    cursor = 0
    for n in range(1, max_section + 1):
        pos = find_section_start(text, n, cursor)
        if pos == -1:
            continue
        positions[n] = pos
        cursor = pos + 1

    found_numbers = sorted(positions.keys())
    parsed = []
    for i, n in enumerate(found_numbers):
        start = positions[n]
        end = positions[found_numbers[i + 1]] if i + 1 < len(found_numbers) else start + 4000
        body = text[start:end].strip()
        parsed.append({
            "id": f"S{n}", "number": n, "chapter": chapter_for_section(n),
            "title": SECTION_TITLES.get(n, f"Section {n}"), "raw_text": body,
            "source_url": DPDP_PDF_URL,
        })
    return parsed


def tag_entities(section):
    title_lower = section["title"].lower()
    chapter_upper = section["chapter"].upper()
    entities = {"obligations": [], "rights": [], "penalties": [], "definitions": []}

    if "right" in title_lower:
        entities["rights"].append(section["title"])
    if "obligation" in title_lower or "duties" in title_lower or "OBLIGATIONS" in chapter_upper:
        entities["obligations"].append(section["title"])
    if "penalt" in title_lower or "PENALTIES" in chapter_upper:
        entities["penalties"].append(section["title"])
    if section["number"] == 2:
        pattern = re.compile(r'["\u201c]([^"\u201d]{2,60})["\u201d]\s+means')
        entities["definitions"] = pattern.findall(section["raw_text"])[:10]

    return entities


def estimate_confidence(section):
    length = len(section["raw_text"])
    if length < 40:
        return 0.4
    score = 0.75 + min(length, 1200) / 1200 * 0.2
    return round(min(score, 0.97), 2)


def is_sensitive(entities):
    return any(len(entities.get(cat, [])) > 0 for cat in SENSITIVE_CATEGORIES)


def commit_section_to_stores(section):
    embedding = embedder.encode(section["raw_text"]).tolist()
    qdrant.upsert(
        collection_name=COLLECTION,
        points=[PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding,
            payload={
                "kg_node_id": section["id"],
                "title": section["title"],
                "text": section["raw_text"],
                "source_url": section["source_url"],
            },
        )],
    )

    with neo4j_driver.session() as session:
        session.run(
            "MERGE (s:Section {id: $id}) SET s.title = $title, s.source_url = $url",
            id=section["id"], title=section["title"], url=section["source_url"],
        )
        entities = section["entities"]
        for label, items in (("Obligation", entities["obligations"]), ("Right", entities["rights"]),
                              ("Penalty", entities["penalties"]), ("Definition", entities["definitions"])):
            for name in items:
                node_id = f"{label.lower()}::{name}"[:200]
                session.run(
                    f"MERGE (e:`{label}` {{id: $id}}) SET e.name = $name",
                    id=node_id, name=name,
                )
                session.run(
                    "MATCH (s:Section {id: $sid}), (e {id: $eid}) MERGE (s)-[:MENTIONS]->(e)",
                    sid=section["id"], eid=node_id,
                )


def ensure_data_ingested():
    if qdrant.collection_exists(COLLECTION):
        return

    print("Creating Qdrant collection...")
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )

    print("Downloading the official DPDP Act PDF...")
    if not os.path.exists(LOCAL_PDF_PATH):
        res = requests.get(DPDP_PDF_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
        res.raise_for_status()
        with open(LOCAL_PDF_PATH, "wb") as f:
            f.write(res.content)

    reader = PdfReader(LOCAL_PDF_PATH)
    raw_text = ""
    for page in reader.pages:
        raw_text += (page.extract_text() or "") + "\n"

    parsed_sections = split_into_sections(raw_text)[:MAX_SECTIONS]
    print(f"Parsed {len(parsed_sections)} sections. Tagging (rule-based, fast) and gating through review...")

    for section in parsed_sections:
        section["entities"] = tag_entities(section)
        confidence = estimate_confidence(section)
        ALL_SECTIONS[section["id"]] = section

        sensitive = is_sensitive(section["entities"])
        needs_review = sensitive or confidence < CONFIDENCE_THRESHOLD

        REVIEW_QUEUE[section["id"]] = {
            "section_id": section["id"], "title": section["title"],
            "confidence": confidence, "sensitive": sensitive,
            "status": "pending_review" if needs_review else "auto_approved",
            "reviewed_by": None, "reviewed_at": None,
        }

        if not needs_review:
            commit_section_to_stores(section)

    approved = sum(1 for e in REVIEW_QUEUE.values() if e["status"] == "auto_approved")
    pending = sum(1 for e in REVIEW_QUEUE.values() if e["status"] == "pending_review")
    print(f"Ingestion complete: {approved} auto-approved and indexed, {pending} pending human review.")


@app.on_event("startup")
def startup_event():
    ensure_data_ingested()


class QueryRequest(BaseModel):
    query: str


class QueryResponse(BaseModel):
    query: str
    status: str
    answer: Optional[str] = None
    citations: List[str] = []
    note: Optional[str] = None


class ReviewDecision(BaseModel):
    section_id: str
    decision: str
    reviewer: str


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/pending-review")
def pending_review():
    return [e for e in REVIEW_QUEUE.values() if e["status"] == "pending_review"]


@app.post("/approve-review-item")
def approve_review_item(decision: ReviewDecision):
    if decision.section_id not in REVIEW_QUEUE:
        raise HTTPException(status_code=404, detail=f"No such section {decision.section_id}")
    if decision.decision not in ("approve", "reject"):
        raise HTTPException(status_code=400, detail="decision must be 'approve' or 'reject'")

    entry = REVIEW_QUEUE[decision.section_id]
    entry["status"] = "approved" if decision.decision == "approve" else "rejected"
    entry["reviewed_by"] = decision.reviewer
    entry["reviewed_at"] = datetime.now(timezone.utc).isoformat()

    if decision.decision == "approve":
        commit_section_to_stores(ALL_SECTIONS[decision.section_id])

    return {"section_id": decision.section_id, "new_status": entry["status"]}


def query_is_high_risk(query):
    lowered = query.lower()
    return any(keyword in lowered for keyword in HIGH_RISK_QUERY_KEYWORDS)


@app.post("/ask", response_model=QueryResponse)
def ask_dpdp_api(request: QueryRequest):
    try:
        ensure_data_ingested()

        cache_key = request.query.strip().lower()
        if cache_key in ANSWER_CACHE:
            return ANSWER_CACHE[cache_key]

        query_vector = embedder.encode(request.query).tolist()
        results = qdrant.query_points(
            collection_name=COLLECTION, query=query_vector, limit=2,
        ).points

        if not results:
            return QueryResponse(query=request.query, status="no_answer",
                                  note="No approved sections matched yet - check /pending-review.")

        context_str = ""
        citations = []
        for res in results:
            node_id = res.payload["kg_node_id"]
            citations.append(node_id)
            cypher = """
            MATCH (s:Section {id: $node_id})-[:MENTIONS]->(related)
            RETURN labels(related)[0] AS type, related.name AS name
            """
            with neo4j_driver.session() as session:
                g_data = session.run(cypher, node_id=node_id).data()
            g_info = "\n".join(f"  - {g['type']}: {g['name']}" for g in g_data)
            context_str += f"\n[{node_id}] {res.payload['text'][:600]}\n{g_info}\n"

        if query_is_high_risk(request.query):
            return QueryResponse(
                query=request.query, status="pending_review", citations=citations,
                note="High-risk question (penalty/obligation/cross-border). Not shown as a final "
                     "answer until a human reviews it - see /pending-review.",
            )

        prompt = f"Answer concisely in 2-3 sentences based on this DPDP Act context:\n{context_str}\n\nQuestion: {request.query}"
        ans = ollama.chat(
            model="qwen2.5:3b",
            messages=[{"role": "user", "content": prompt}],
            options={"num_predict": 200, "num_ctx": 1024},  # caps response length + context window for speed
            keep_alive="30m",  # keeps the model loaded in memory between questions, avoids reload latency
        )

        response = QueryResponse(
            query=request.query, status="answered",
            answer=ans["message"]["content"], citations=citations,
        )
        ANSWER_CACHE[cache_key] = response
        return response
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# 6. Write the Streamlit frontend (`app.py`)

Chat interface plus a live "Pending Human Review" panel in the sidebar - approve or reject sections
right from the app; approved sections become searchable immediately.

In [ ]:
%%writefile app.py
import streamlit as st
import requests

API_URL = "http://localhost:8000"

st.set_page_config(page_title="DPDP Act Assistant", page_icon="⚖️")
st.title("India DPDP Act 2023 - GraphRAG Assistant")
st.markdown("Ask questions about obligations, rights, and penalties under the DPDP Act.")

with st.sidebar:
    st.header("Pending Human Review")
    st.caption("Sections flagged as sensitive or low-confidence. Nothing here is searchable until approved.")
    try:
        pending = requests.get(f"{API_URL}/pending-review", timeout=10).json()
    except requests.exceptions.RequestException:
        pending = []
        st.warning("Backend is starting up - refresh in a moment.")

    if not pending:
        st.info("No sections waiting for review right now.")
    else:
        for item in pending:
            with st.expander(f"{item['section_id']} - {item['title']}"):
                st.write(f"Confidence: {item['confidence']:.2f}")
                st.write(f"Sensitive: {item['sensitive']}")
                col1, col2 = st.columns(2)
                if col1.button("Approve", key=f"approve_{item['section_id']}"):
                    requests.post(f"{API_URL}/approve-review-item", json={
                        "section_id": item["section_id"], "decision": "approve", "reviewer": "demo_reviewer",
                    }, timeout=30)
                    st.rerun()
                if col2.button("Reject", key=f"reject_{item['section_id']}"):
                    requests.post(f"{API_URL}/approve-review-item", json={
                        "section_id": item["section_id"], "decision": "reject", "reviewer": "demo_reviewer",
                    }, timeout=30)
                    st.rerun()

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("Ask a question about the DPDP Act..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Searching the knowledge graph and generating an answer..."):
            try:
                response = requests.post(f"{API_URL}/ask", json={"query": prompt}, timeout=300)
                if response.status_code == 200:
                    data = response.json()
                    if data["status"] == "answered":
                        answer = data["answer"]
                        st.markdown(answer)
                        if data.get("citations"):
                            st.caption(f"Referenced sections: {', '.join(data['citations'])}")
                        st.session_state.messages.append({"role": "assistant", "content": answer})
                    elif data["status"] == "pending_review":
                        note = data.get("note", "This question needs human review before an answer is shown.")
                        st.warning(note)
                        st.session_state.messages.append({"role": "assistant", "content": note})
                    else:
                        note = data.get("note", "No answer available yet.")
                        st.info(note)
                        st.session_state.messages.append({"role": "assistant", "content": note})
                else:
                    st.error(f"API error ({response.status_code}): {response.text}")
            except requests.exceptions.ReadTimeout:
                st.error("Server timeout - the local model took too long to respond.")
            except requests.exceptions.ConnectionError:
                st.error("Backend is still starting up - wait a few seconds and try again.")


## 7. Launch the backend and frontend

Logs are redirected to `uvicorn.log` and `streamlit.log`. Startup should now take well under a
minute (no LLM calls happen until you actually ask a question).

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn"])
subprocess.run(["pkill", "-f", "streamlit"])
time.sleep(2)

uvicorn_log = open("uvicorn.log", "w")
streamlit_log = open("streamlit.log", "w")

print("Starting FastAPI backend on port 8000 (logs -> uvicorn.log)...")
subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=uvicorn_log, stderr=subprocess.STDOUT,
)

print("Starting Streamlit frontend on port 8501 (logs -> streamlit.log)...")
subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501", "--server.headless", "true",
        "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
    ],
    stdout=streamlit_log, stderr=subprocess.STDOUT,
)

print("Both processes launched. Startup is now fast (fetch + parse + rule-based tagging, no LLM")
print("calls) - should be ready in under a minute. Next cell waits for it and shows live progress.")


## 8. Wait for the backend, then test it (inside Colab, no tunnel needed yet)

Polls `/health`, and prints the tail of `uvicorn.log` every 15 seconds so you can see actual
progress instead of a silent wait. If it still doesn't come up, the full log is printed for
debugging.

In [ ]:
import requests as _requests
import os
import time as _time


def tail_log(path, n=60):
    if not os.path.exists(path):
        return "(no log file yet)"
    with open(path) as f:
        lines = f.readlines()
    return "".join(lines[-n:]) if lines else "(log file is empty so far)"


def wait_for_backend(max_wait_seconds=180, poll_interval=5):
    waited = 0
    while waited < max_wait_seconds:
        try:
            r = _requests.get("http://localhost:8000/health", timeout=5)
            if r.status_code == 200:
                print(f"Backend is up after about {waited}s.")
                return True
        except _requests.exceptions.RequestException:
            pass
        if waited % 15 == 0:
            print(f"  ({waited}s elapsed) latest log lines:")
            print("    " + tail_log("uvicorn.log", 3).replace("\n", "\n    "))
        _time.sleep(poll_interval)
        waited += poll_interval
    return False


print("Waiting for the backend (should be well under a minute now)...")
if not wait_for_backend():
    print()
    print("Backend did not come up in time. Full uvicorn.log:")
    print("-" * 70)
    print(tail_log("uvicorn.log"))
    print("-" * 70)
    print("Paste this log back to fix the actual cause - re-running cell 7 then this cell after a fix.")
else:
    print()
    print("Health check:")
    print(_requests.get("http://localhost:8000/health", timeout=10).json())

    print()
    print("Pending review queue (count):")
    pending = _requests.get("http://localhost:8000/pending-review", timeout=10).json()
    print(f"{len(pending)} sections pending review")

    print()
    print("Normal question:")
    r = _requests.post("http://localhost:8000/ask", json={"query": "What are the grounds for processing personal data?"}, timeout=300)
    print(r.json())

    print()
    print("High-risk question (should come back pending_review, not a final answer):")
    r = _requests.post("http://localhost:8000/ask", json={"query": "What penalty applies if a company fails to report a data breach?"}, timeout=300)
    print(r.json())


## 8b. (Optional) view the raw logs any time

In [ ]:
print("--- uvicorn.log (last 60 lines) ---")
print(tail_log("uvicorn.log", 60))
print()
print("--- streamlit.log (last 30 lines) ---")
print(tail_log("streamlit.log", 30))


## 9. Open a public URL (free, no signup - `cloudflared`)

`localtunnel` is known to have trouble proxying Streamlit's dynamically-imported JS chunks
correctly (a "Failed to fetch dynamically imported module" error in the browser is the telltale
sign). `cloudflared`'s free quick tunnels handle this reliably, and there's no interstitial warning
page or password step at all - the URL just works.

In [ ]:
import re as _re

print("Opening a Cloudflare quick tunnel to the Streamlit app on port 8501...")
tunnel_log = open("cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    with open("cloudflared.log") as f:
        log_content = f.read()
    match = _re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"Your app is live at: {public_url}")
    print("No password, no signup - open it directly. This process keeps running in the")
    print("background to keep the tunnel open; you do not need to leave this cell running.")
else:
    print("Tunnel URL not found yet - here is cloudflared.log so far:")
    with open("cloudflared.log") as f:
        print(f.read())
    print("Re-run this cell if the URL did not appear - it usually takes a few seconds.")


## Project summary

**What's real in this demo:** the DPDP Act text, fetched live from the official government PDF every
time this notebook runs (not hardcoded); a real Neo4j graph and real Qdrant vector index; a real
local LLM answering questions; a working human-review gate that actually blocks unapproved content
from being searchable; and a public, shareable URL.

**What's simplified, and clearly marked as such throughout:** rule-based rather than LLM-based entity
tagging (Section 5 of `main.py`), a single Act rather than all 70+ jurisdictions in the original
project scope, and ephemeral rather than persistent infrastructure. None of these are hidden - each
is called out at the point in the notebook where it matters.

## Where this would grow for a production system

1. Replace rule-based tagging with a real LLM extraction agent run as an **offline batch job**
   (not blocking server startup) - the review-queue mechanics here plug in directly, only the
   source of the confidence score changes.
2. Persist Neo4j and Qdrant outside the Colab container - both live in `/content` here and are
   wiped when the session ends.
3. Also ingest the **Digital Personal Data Protection Rules, 2025** (notified 13 November 2025) -
   this only covers the original 2023 Act text.
4. Add amendment monitoring: stage detected law changes for human confirmation before they update
   the graph, same `review_queue` pattern extended.
5. Repeat this same pattern per country to build out the full multi-jurisdiction graph described in
   the original project scope (70+ jurisdictions, DPDP/GDPR/CCPA/LGPD/PIPL and others).
6. Swap `qwen2.5:3b` for a larger Ollama model (e.g. `qwen2.5:7b`, `llama3.1:8b`) if answer precision
   needs to improve - the "Section 41 vs Section 42" mislabeling example above is a small-model
   limitation, not an architectural one.
